### Cobb angle prediction

In [9]:
import os
import pandas as pd
import numpy as np
import trimesh
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [10]:
excel_path = r"D:/Professional/Projects/Aiims/Gait/Data/project work/PROJECTBook1.xlsx"
df_cobb = pd.read_excel(excel_path)

# Drop where we don't have cobb angle

df_cobb_s = df_cobb[~df_cobb["COBB'S"].isna()]

# Assuming 'PATIENT_NAME' matches folder name, 'COBB\'S' has Cobb angle
patient_to_cobb = dict(zip(df_cobb_s['NAME'], df_cobb_s["COBB'S"]))


In [3]:
import trimesh
import os

def extract_torso_mesh(input_path, output_path=None, keep_largest=False):
    """
    Extract the torso mesh from an STL file by removing side panels/noise.

    Parameters
    ----------
    input_path : str
        Path to input STL file.
    output_path : str, optional
        Path to save cleaned torso STL. If None, will save as <input_name>_torso.stl
    keep_largest : bool
        If True, keeps only the largest mesh (by surface area).
        If False, applies heuristics (center + non-flat meshes).
    
    Returns
    -------
    torso_mesh : trimesh.Trimesh
        The cleaned torso mesh.
    """
    
    # Load STL
    mesh = trimesh.load(input_path)

    # Split into disconnected components
    submeshes = mesh.split(only_watertight=False)
    print(f"[INFO] Found {len(submeshes)} submeshes in {os.path.basename(input_path)}")

    if keep_largest:
        torso_mesh = max(submeshes, key=lambda m: m.area)
    else:
        # Heuristic: keep meshes near center and not flat
        centered = [m for m in submeshes if abs(m.centroid[0]) < 150]  # adjust threshold if needed
        torso_mesh = max(centered, key=lambda m: m.area) if centered else max(submeshes, key=lambda m: m.area)

    # Save cleaned STL
    if output_path is None:
        output_path = input_path.replace(".stl", "_torso.stl")
    torso_mesh.export(output_path)
    
    print(f"[INFO] Saved torso mesh -> {output_path}")
    return torso_mesh


In [4]:
import open3d as o3d
import numpy as np
import os

def clean_mesh_keep_torso(stl_path, save_dir="cleaned_meshes"):
    # Load mesh
    mesh = o3d.io.read_triangle_mesh(stl_path)
    mesh.compute_vertex_normals()
    
    # Sample points from mesh
    pcd = mesh.sample_points_poisson_disk(number_of_points=50000)
    
    # Cluster points
    labels = np.array(pcd.cluster_dbscan(eps=15, min_points=100))
    
    torso_cloud = None
    max_height = 0
    
    for cluster_id in np.unique(labels):
        if cluster_id == -1:
            continue
        cluster_points = pcd.select_by_index(np.where(labels == cluster_id)[0])
        
        bbox = cluster_points.get_axis_aligned_bounding_box()
        height = bbox.get_extent()[2]  # vertical size
        
        if height > max_height and height > 30:  # keep largest tall object
            max_height = height
            torso_cloud = cluster_points
    
    if torso_cloud is None:
        return None
    
    # Convert torso point cloud back to mesh (Ball Pivoting Reconstruction)
    distances = torso_cloud.compute_nearest_neighbor_distance()
    avg_dist = np.mean(distances)
    radius = 2.5 * avg_dist
    
    mesh_torso = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        torso_cloud, o3d.utility.DoubleVector([radius, radius * 2])
    )
    
    # Save result
    os.makedirs(save_dir, exist_ok=True)
    out_path = os.path.join(save_dir, os.path.basename(stl_path))
    o3d.io.write_triangle_mesh(out_path, mesh_torso)
    
    return out_path


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [5]:
extract_torso_mesh(input_path="D:/Professional/Projects/Aiims/Gait/Data/Sample/2_Gracy_Singh/130_1.stl")

[INFO] Found 1216 submeshes in 130_1.stl
[INFO] Saved torso mesh -> D:/Professional/Projects/Aiims/Gait/Data/Sample/2_Gracy_Singh/130_1_torso.stl


<trimesh.Trimesh(vertices.shape=(126611, 3), faces.shape=(244097, 3))>

In [6]:
clean_mesh_keep_torso(stl_path="D:/Professional/Projects/Aiims/Gait/Data/Sample/2_Gracy_Singh/130_1.stl")

In [16]:
import open3d as o3d
import numpy as np

def crop_torso_with_bbox(stl_path, save_path,
                         x_range=(-100, 100),
                         y_range=(-100, 100),
                         z_range=(0, 600)):
    """
    Crops STL mesh to keep only torso region using a fixed bounding box.
    
    Args:
        stl_path (str): Input STL file path.
        save_path (str): Output cropped STL path.
        x_range, y_range, z_range (tuple): Bounds in each axis (mm).
    
    Returns:
        o3d.geometry.TriangleMesh: Cropped mesh
    """
    # Load mesh
    mesh = o3d.io.read_triangle_mesh(stl_path)
    mesh.compute_vertex_normals()

    # Define bounding box
    bbox = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=(x_range[0], y_range[0], z_range[0]),
        max_bound=(x_range[1], y_range[1], z_range[1])
    )

    # Crop mesh
    torso_mesh = mesh.crop(bbox)

    # Save cropped STL
    o3d.io.write_triangle_mesh(save_path, torso_mesh)
    
    return torso_mesh

# Example usage
# torso = crop_torso_with_bbox("input.stl", "cropped_torso.stl",
#                              x_range=(-120, 120),
#                              y_range=(-80, 80),
#                              z_range=(50, 500))


In [17]:
crop_torso_with_bbox(stl_path="D:/Professional/Projects/Aiims/Gait/Data/Sample/2_Gracy_Singh/130_1.stl",
                     save_path="D:/Professional/Projects/Aiims/Gait/Data/cleaned_surface_topography/cropped_130_1.stl")

[Open3D WARNING] Write STL failed: unable to open file.


TriangleMesh with 0 points and 0 triangles.

In [18]:
import open3d as o3d
import os

def crop_torso_with_bbox_auto_save(stl_path,
                                  x_range=(-100, 100),
                                  y_range=(-100, 100),
                                  z_range=(0, 600)):
    """
    Crops STL mesh to keep only torso region using a fixed bounding box.
    Automatically saves in a 'cropped_meshes' folder next to the original file.
    
    Args:
        stl_path (str): Input STL file path.
        x_range, y_range, z_range (tuple): Bounds in each axis (mm).
    
    Returns:
        o3d.geometry.TriangleMesh: Cropped mesh
        str: Saved file path
    """
    # Load mesh
    mesh = o3d.io.read_triangle_mesh(stl_path)
    mesh.compute_vertex_normals()

    # Define bounding box
    bbox = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=(x_range[0], y_range[0], z_range[0]),
        max_bound=(x_range[1], y_range[1], z_range[1])
    )

    # Crop mesh
    torso_mesh = mesh.crop(bbox)

    # Create new folder for cropped meshes
    folder = os.path.join(os.path.dirname(stl_path), "cropped_meshes")
    os.makedirs(folder, exist_ok=True)

    # Save cropped STL with same filename
    filename = os.path.basename(stl_path)
    save_path = os.path.join(folder, f"cropped_{filename}")
    o3d.io.write_triangle_mesh(save_path, torso_mesh)
    
    print(f"Cropped mesh saved at: {save_path}")
    return torso_mesh, save_path

# Example usage
torso, saved_path = crop_torso_with_bbox_auto_save(
    "D:/Professional/Projects/Aiims/Gait/Data/Sample/2_Gracy_Singh/130_1.stl"
)


[Open3D WARNING] Write STL failed: compute normals first.
Cropped mesh saved at: D:/Professional/Projects/Aiims/Gait/Data/Sample/2_Gracy_Singh\cropped_meshes\cropped_130_1.stl


In [10]:
import open3d as o3d
import os
import numpy as np

def crop_torso_auto_save(stl_path, z_lower=0.2, z_upper=0.8, xy_margin=0.05):
    """
    Automatically crops the torso region from an STL mesh and saves it.
    
    Args:
        stl_path (str): Input STL file path.
        z_lower, z_upper (float): Fraction of height to keep (0-1) for torso.
        xy_margin (float): Fraction of width/depth to trim from sides.
    
    Returns:
        o3d.geometry.TriangleMesh: Cropped mesh.
        str: Saved STL file path.
    """
    # Load mesh
    mesh = o3d.io.read_triangle_mesh(stl_path)
    if mesh.is_empty():
        raise ValueError("Mesh is empty!")
    
    # Get mesh bounds
    min_bound = mesh.get_min_bound()
    max_bound = mesh.get_max_bound()
    dims = max_bound - min_bound

    # Define torso bounding box
    x_range = (min_bound[0] + xy_margin*dims[0], max_bound[0] - xy_margin*dims[0])
    y_range = (min_bound[1] + xy_margin*dims[1], max_bound[1] - xy_margin*dims[1])
    z_range = (min_bound[2] + z_lower*dims[2], min_bound[2] + z_upper*dims[2])

    bbox = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=(x_range[0], y_range[0], z_range[0]),
        max_bound=(x_range[1], y_range[1], z_range[1])
    )

    # Crop mesh
    torso_mesh = mesh.crop(bbox)
    if torso_mesh.is_empty():
        raise ValueError("Cropped mesh is empty! Adjust bounds.")

    # Compute normals for cropped mesh
    torso_mesh.compute_vertex_normals()

    # Clean mesh
    torso_mesh.remove_duplicated_vertices()
    torso_mesh.remove_duplicated_triangles()
    torso_mesh.remove_non_manifold_edges()
    torso_mesh.remove_degenerate_triangles()

    # Auto-create folder and save
    folder = os.path.join(os.path.dirname(stl_path), "cropped_meshes")
    os.makedirs(folder, exist_ok=True)
    filename = os.path.basename(stl_path)
    save_path = os.path.join(folder, f"cropped_{filename}")

    o3d.io.write_triangle_mesh(save_path, torso_mesh)
    print(f"Cropped torso saved at: {save_path}")

    return torso_mesh, save_path

# Example usage
torso, saved_path = crop_torso_auto_save(
    "D:/Professional/Projects/Aiims/Gait/Data/Sample/2_Gracy_Singh/130_1.stl"
)


Cropped torso saved at: D:/Professional/Projects/Aiims/Gait/Data/Sample/2_Gracy_Singh\cropped_meshes\cropped_130_1.stl


In [3]:
import open3d as o3d
import os
import numpy as np

def crop_torso(stl_path, 
               x_range=None, y_range=None, z_range=None,
               z_lower_frac=0.2, z_upper_frac=0.8,
               xy_margin_frac=0.05):
    """
    Crops torso region from STL mesh, either automatically or using manual bounds.
    
    Args:
        stl_path (str): Input STL file path.
        x_range, y_range, z_range (tuple or None): Manual bounding box in mm.
        z_lower_frac, z_upper_frac (float): Fraction of height to keep if z_range is None.
        xy_margin_frac (float): Fraction to trim from sides if x/y ranges are None.
    
    Returns:
        o3d.geometry.TriangleMesh: Cropped mesh.
        str: Saved STL path.
    """
    # Load mesh
    mesh = o3d.io.read_triangle_mesh(stl_path)
    if mesh.is_empty():
        raise ValueError("Mesh is empty!")
    
    # Compute bounds for auto mode
    min_bound = mesh.get_min_bound()
    max_bound = mesh.get_max_bound()
    dims = max_bound - min_bound

    # Determine ranges
    if x_range is None:
        x_range = (min_bound[0] + xy_margin_frac*dims[0], max_bound[0] - xy_margin_frac*dims[0])
    if y_range is None:
        y_range = (min_bound[1] + xy_margin_frac*dims[1], max_bound[1] - xy_margin_frac*dims[1])
    if z_range is None:
        z_range = (min_bound[2] + z_lower_frac*dims[2], min_bound[2] + z_upper_frac*dims[2])

    # Crop bounding box
    bbox = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=(x_range[0], y_range[0], z_range[0]),
        max_bound=(x_range[1], y_range[1], z_range[1])
    )
    
    torso_mesh = mesh.crop(bbox)
    if torso_mesh.is_empty():
        raise ValueError("Cropped mesh is empty! Adjust bounds.")

    # Compute normals for cropped mesh
    torso_mesh.compute_vertex_normals()

    # Clean mesh
    torso_mesh.remove_duplicated_vertices()
    torso_mesh.remove_duplicated_triangles()
    torso_mesh.remove_non_manifold_edges()
    torso_mesh.remove_degenerate_triangles()

    # Save automatically
    folder = os.path.join(os.path.dirname(stl_path), "cropped_meshes")
    os.makedirs(folder, exist_ok=True)
    filename = os.path.basename(stl_path)
    save_path = os.path.join(folder, f"cropped_{filename}")
    o3d.io.write_triangle_mesh(save_path, torso_mesh)
    
    print(f"Cropped torso saved at: {save_path}")
    return torso_mesh, save_path


In [14]:
torso_mesh, saved_path = crop_torso(
    "D:/Professional/Projects/Aiims/Gait/Data/Sample/2_Gracy_Singh/130_1.stl",
    x_range=(-120, 120),  # slightly smaller than full width
    y_range=(-80, 80),    # focus central depth
    z_range=(300, 1200)   # torso height (avoid legs and head)
)


ValueError: Cropped mesh is empty! Adjust bounds.

In [5]:
import open3d as o3d

mesh = o3d.io.read_triangle_mesh(
    "D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi/150_1.stl"
)
print("Mesh min bound:", mesh.get_min_bound())
print("Mesh max bound:", mesh.get_max_bound())
print("Mesh dimensions:", mesh.get_max_bound() - mesh.get_min_bound())


Mesh min bound: [-0.74804688 -0.74804688 -1.99804688]
Mesh max bound: [ 0.74804688  0.74804688 -0.79790902]
Mesh dimensions: [1.49609375 1.49609375 1.20013785]


In [6]:
torso_mesh, saved_path = crop_torso(
    "D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi/150_1.stl",
    x_range=(-0.5, 0.5),
    y_range=(-0.698, 0.698),
    z_range=(-1.8, -1.06445312)
)

print("Cropped mesh saved at:", saved_path)
o3d.visualization.draw_geometries([torso_mesh])


Cropped torso saved at: D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi\cropped_meshes\cropped_150_1.stl
Cropped mesh saved at: D:/Professional/Projects/Aiims/Gait/Data/project work/Surface topography data/010824/3_Gargi\cropped_meshes\cropped_150_1.stl


In [15]:
def extract_features_from_stl(file_path):
    """
    Extract handcrafted features from an STL file using trimesh.
    """
    mesh_obj = trimesh.load(file_path)
    bbox = mesh_obj.bounding_box.extents
    feature_dict = {
        "surface_area": mesh_obj.area,
        "volume": mesh_obj.volume,
        "bbox_x": bbox[0],
        "bbox_y": bbox[1],
        "bbox_z": bbox[2],
        "compactness": mesh_obj.volume / (mesh_obj.area ** 1.5 + 1e-6),
        "elongation": max(bbox) / (min(bbox) + 1e-6)
    }
    return feature_dict


In [16]:
# Extract features from STL

def extract_features_from_stl(file_path):
    """
    Extract handcrafted features from an STL file using trimesh.
    """
    mesh_obj = trimesh.load(file_path)
    bbox = mesh_obj.bounding_box.extents
    feature_dict = {
        "surface_area": mesh_obj.area,
        "volume": mesh_obj.volume,
        "bbox_x": bbox[0],
        "bbox_y": bbox[1],
        "bbox_z": bbox[2],
        "compactness": mesh_obj.volume / (mesh_obj.area ** 1.5 + 1e-6),
        "elongation": max(bbox) / (min(bbox) + 1e-6)
    }
    return feature_dict


In [37]:
from rapidfuzz import process, fuzz
from tqdm import tqdm  # <-- tqdm for progress tracking
import os
import pandas as pd

# Function to clean folder name
def clean_patient_name(folder_name):
    if '_' in folder_name:
        return folder_name.split('_', 1)[1]  # Keep everything after first underscore
    return folder_name

# Function for fuzzy matching
def match_patient_name(clean_name, patient_list, threshold=80):
    match, score, _ = process.extractOne(clean_name, patient_list, scorer=fuzz.token_sort_ratio)
    if score >= threshold:
        return match
    return None

# List to store problematic files
problematic_files = []

# Updated loop with tqdm
root_dir = r"D:/Professional/Projects/Aiims/Gait/Data/project work/Surface Topography Data"
data_rows = []
excel_names = df_cobb_s['NAME'].tolist()

date_folders = [f for f in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, f))]

for date_folder in tqdm(date_folders, desc="Date Folders"):
    date_path = os.path.join(root_dir, date_folder)
    patient_folders = [pf for pf in os.listdir(date_path) if os.path.isdir(os.path.join(date_path, pf))]
    
    for patient_folder in tqdm(patient_folders, desc=f"Patients in {date_folder}", leave=False):
        patient_path = os.path.join(date_path, patient_folder)
        # Clean folder name
        clean_name = clean_patient_name(patient_folder)
        # Fuzzy match
        matched_name = match_patient_name(clean_name, excel_names)
        if matched_name is None:
            continue
        cobb_angle = patient_to_cobb[matched_name]

        stl_files = [sf for sf in os.listdir(patient_path) if sf.endswith(".stl")]
        for stl_file in tqdm(stl_files, desc=f"STL files for {matched_name}", leave=False):
            stl_path = os.path.join(patient_path, stl_file)
            try:
                features = extract_features_from_stl(stl_path)
                if features is None:
                    problematic_files.append((stl_path, "Empty mesh"))
                    continue
                features['patient_id'] = matched_name
                features['stl_file'] = stl_file
                features['cobb_angle'] = cobb_angle
                data_rows.append(features)
            except Exception as e:
                problematic_files.append((stl_path, str(e)))
                continue

# Create dataframe
df = pd.DataFrame(data_rows)
print("Total STL samples:", len(df))
df.head()

# Show problematic files
if problematic_files:
    print("\nProblematic STL files:")
    for fpath, reason in problematic_files:
        print(f"{fpath} -> {reason}")


Date Folders:   0%|          | 0/5 [00:00<?, ?it/s]































































Date Folders:  20%|██        | 1/5 [00:43<02:54, 43.61s/it]































Date Folders:  40%|████      | 2/5 [01:01<01:25, 28.41s/it]






























Date Folders:  60%|██████    | 3/5 [01:21<00:49, 24.69s/it]































Date Folders: 100%|██████████| 5/5 [01:42<00:00, 20.48s/it]

Total STL samples: 149

Problematic STL files:
D:/Professional/Projects/Aiims/Gait/Data/project work/Surface Topography Data\080824\1_Ayan_Saifi\130_5.stl -> 'NoneType' object has no attribute 'mean'


In [44]:
df.isnull().sum()


surface_area    0
volume          0
bbox_x          0
bbox_y          0
bbox_z          0
compactness     0
elongation      0
patient_id      0
stl_file        0
cobb_angle      0
dtype: int64

In [45]:
df.head()

,surface_area,volume,bbox_x,bbox_y,bbox_z,compactness,elongation,patient_id,stl_file,cobb_angle
0,1.600360,-0.252763,1.496094,1.496094,1.269531,-0.124849,1.178461,Gracy Singh,100_1.stl,44.0
1,1.613899,-0.252456,1.496094,1.496094,1.269531,-0.123132,1.178461,Gracy Singh,100_2.stl,44.0
2,1.617845,-0.254985,1.496094,1.496094,1.328125,-0.123911,1.126470,Gracy Singh,100_3.stl,44.0
3,1.597820,-0.254835,1.496094,1.496094,1.328125,-0.126173,1.126470,Gracy Singh,100_4.stl,44.0
4,1.592781,-0.253951,1.496094,1.496094,1.328125,-0.126333,1.126470,Gracy Singh,100_5.stl,44.0


In [48]:
df['cobb_angle'].unique()

array([44., 67., 68., 30., 61.])

### Predict cobb angle

In [51]:
# Split into train and test

from sklearn.model_selection import GroupShuffleSplit

X = df.drop(columns=['patient_id', 'stl_file', 'cobb_angle'])
y = df['cobb_angle']
groups = df['patient_id']  # ensure patient-level split

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


In [52]:
# Standardize features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [53]:
# Train SVM model

from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr.fit(X_train_scaled, y_train)

y_pred = svr.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))


MAE: 19.27876984858524
R2: 0.0


### Train SVM at patient level

In [61]:
# -------------------------------
# Step 1: Aggregate STL features per patient (numeric columns only)
# -------------------------------
numeric_cols = df.select_dtypes(include='number').columns.tolist()  # Only numeric
patient_features = df.groupby('patient_id')[numeric_cols].mean().reset_index()

# Separate features and target
X = patient_features.drop(columns=['patient_id', 'cobb_angle'], errors='ignore')  # exclude target & ID
y = patient_features['cobb_angle']

# -------------------------------
# Step 2: Train/test split at patient level (keep all STL files of a patient together)
# -------------------------------
from sklearn.model_selection import GroupShuffleSplit
groups = patient_features['patient_id']  # patient-level grouping

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=44)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# -------------------------------
# Step 3: Scale features
# -------------------------------
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------
# Step 4: Train SVM regressor
# -------------------------------
from sklearn.svm import SVR
svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr.fit(X_train_scaled, y_train)

# -------------------------------
# Step 5: Predict and evaluate
# -------------------------------
from sklearn.metrics import mean_absolute_error, r2_score
y_pred = svr.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

# Optional: inspect predictions vs true values
import pandas as pd
results = pd.DataFrame({
    'patient_id': patient_features.iloc[test_idx]['patient_id'],
    'true_cobb': y_test,
    'pred_cobb': y_pred
})
print(results)


MAE: 31.539781662072045
R2: nan
   patient_id  true_cobb  pred_cobb
0  Ayan Saifi       30.0  61.539782


c:\Users\amito\anaconda3\envs\ais\lib\site-packages\sklearn\metrics\_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [58]:
mean_absolute_error(y_test, y_pred)

18.181122179451723

### Maximize feature extraction

In [62]:
from rapidfuzz import process, fuzz
from tqdm import tqdm
import os
import pandas as pd
import numpy as np
import trimesh

# -------------------------------
# Enhanced feature extraction
# -------------------------------
def extract_features_from_stl(stl_path, num_slices=10):
    try:
        mesh_obj = trimesh.load(stl_path)
    except:
        return None  # Skip unreadable files
    
    if mesh_obj.is_empty:
        return None
    
    # Global features
    area = mesh_obj.area
    volume = mesh_obj.volume
    bbox = mesh_obj.bounding_box.extents
    compactness = volume / (area ** 1.5 + 1e-6)
    elongation = max(bbox) / (min(bbox) + 1e-6)
    
    # Bounding box ratios
    bbox_xy = bbox[0]/bbox[1]
    bbox_xz = bbox[0]/bbox[2]
    bbox_yz = bbox[1]/bbox[2]
    
    # Slice-based features
    z_min, z_max = mesh_obj.bounds[:,2]
    slice_heights = np.linspace(z_min, z_max, num_slices)
    
    slice_widths = []
    slice_depths = []
    
    for z in slice_heights:
        slice_pts = mesh_obj.vertices[(mesh_obj.vertices[:,2] >= z) & (mesh_obj.vertices[:,2] < z + (z_max-z_min)/num_slices)]
        if len(slice_pts) == 0:
            continue
        slice_widths.append(slice_pts[:,0].max() - slice_pts[:,0].min())
        slice_depths.append(slice_pts[:,1].max() - slice_pts[:,1].min())
    
    slice_features = {
        'slice_width_mean': np.mean(slice_widths) if slice_widths else 0,
        'slice_width_std': np.std(slice_widths) if slice_widths else 0,
        'slice_depth_mean': np.mean(slice_depths) if slice_depths else 0,
        'slice_depth_std': np.std(slice_depths) if slice_depths else 0,
    }
    
    feature_dict = {
        'surface_area': area,
        'volume': volume,
        'bbox_x': bbox[0],
        'bbox_y': bbox[1],
        'bbox_z': bbox[2],
        'compactness': compactness,
        'elongation': elongation,
        'bbox_xy': bbox_xy,
        'bbox_xz': bbox_xz,
        'bbox_yz': bbox_yz,
        **slice_features
    }
    
    return feature_dict

# -------------------------------
# Patient name utilities
# -------------------------------
def clean_patient_name(folder_name):
    if '_' in folder_name:
        return folder_name.split('_', 1)[1]
    return folder_name

def match_patient_name(clean_name, patient_list, threshold=80):
    match, score, _ = process.extractOne(clean_name, patient_list, scorer=fuzz.token_sort_ratio)
    if score >= threshold:
        return match
    return None

# -------------------------------
# Main STL processing loop
# -------------------------------
root_dir = r"D:/Professional/Projects/Aiims/Gait/Data/project work/Surface Topography Data"
excel_names = df_cobb_s['NAME'].tolist()
problematic_files = []
data_rows = []

date_folders = [f for f in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, f))]

for date_folder in tqdm(date_folders, desc="Date Folders"):
    date_path = os.path.join(root_dir, date_folder)
    patient_folders = [pf for pf in os.listdir(date_path) if os.path.isdir(os.path.join(date_path, pf))]
    
    for patient_folder in tqdm(patient_folders, desc=f"Patients in {date_folder}", leave=False):
        patient_path = os.path.join(date_path, patient_folder)
        clean_name = clean_patient_name(patient_folder)
        matched_name = match_patient_name(clean_name, excel_names)
        if matched_name is None:
            continue
        cobb_angle = patient_to_cobb[matched_name]

        stl_files = [sf for sf in os.listdir(patient_path) if sf.endswith(".stl")]
        for stl_file in tqdm(stl_files, desc=f"STL files for {matched_name}", leave=False):
            stl_path = os.path.join(patient_path, stl_file)
            try:
                features = extract_features_from_stl(stl_path)
                if features is None:
                    problematic_files.append((stl_path, "Empty or unreadable mesh"))
                    continue
                features['patient_id'] = matched_name
                features['stl_file'] = stl_file
                features['cobb_angle'] = cobb_angle
                data_rows.append(features)
            except Exception as e:
                problematic_files.append((stl_path, str(e)))
                continue

# -------------------------------
# Create dataframe
# -------------------------------
df_features = pd.DataFrame(data_rows)
print("Total STL samples:", len(df_features))
df_features.head()

# -------------------------------
# Show problematic files
# -------------------------------
if problematic_files:
    print("\nProblematic STL files:")
    for fpath, reason in problematic_files:
        print(f"{fpath} -> {reason}")


Date Folders:   0%|          | 0/5 [00:00<?, ?it/s]































































Date Folders:  20%|██        | 1/5 [00:44<02:57, 44.35s/it]































Date Folders:  40%|████      | 2/5 [00:59<01:22, 27.40s/it]






























Date Folders:  60%|██████    | 3/5 [01:16<00:45, 22.65s/it]































Date Folders: 100%|██████████| 5/5 [01:33<00:00, 18.66s/it]

Total STL samples: 149

Problematic STL files:
D:/Professional/Projects/Aiims/Gait/Data/project work/Surface Topography Data\080824\1_Ayan_Saifi\130_5.stl -> Empty or unreadable mesh


In [64]:
#SVM at stl level

# Split into train and test

from sklearn.model_selection import GroupShuffleSplit

X = df.drop(columns=['patient_id', 'stl_file', 'cobb_angle'])
y = df['cobb_angle']
groups = df['patient_id']  # ensure patient-level split

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Standardize features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train SVM model

from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr.fit(X_train_scaled, y_train)

y_pred = svr.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))




MAE: 19.27876984858524
R2: 0.0


In [68]:
# -------------------------------
# Step 1: Aggregate STL features per patient (numeric columns only)
# -------------------------------
numeric_cols = df.select_dtypes(include='number').columns.tolist()  # Only numeric
patient_features = df.groupby('patient_id')[numeric_cols].mean().reset_index()

# Separate features and target
X = patient_features.drop(columns=['patient_id', 'cobb_angle'], errors='ignore')  # exclude target & ID
y = patient_features['cobb_angle']

# -------------------------------
# Step 2: Train/test split at patient level (keep all STL files of a patient together)
# -------------------------------
from sklearn.model_selection import GroupShuffleSplit
groups = patient_features['patient_id']  # patient-level grouping

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=48)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# -------------------------------
# Step 3: Scale features
# -------------------------------
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------
# Step 4: Train SVM regressor
# -------------------------------
from sklearn.svm import SVR
svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr.fit(X_train_scaled, y_train)

# -------------------------------
# Step 5: Predict and evaluate
# -------------------------------
from sklearn.metrics import mean_absolute_error, r2_score
y_pred = svr.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

# Optional: inspect predictions vs true values
import pandas as pd
results = pd.DataFrame({
    'patient_id': patient_features.iloc[test_idx]['patient_id'],
    'true_cobb': y_test,
    'pred_cobb': y_pred
})
print(results)


MAE: 19.103235760519674
R2: nan
    patient_id  true_cobb  pred_cobb
2  Gracy Singh       44.0  63.103236


c:\Users\amito\anaconda3\envs\ais\lib\site-packages\sklearn\metrics\_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [70]:
df

,surface_area,volume,bbox_x,bbox_y,bbox_z,compactness,elongation,patient_id,stl_file,cobb_angle
0,1.600360,-0.252763,1.496094,1.496094,1.269531,-0.124849,1.178461,Gracy Singh,100_1.stl,44.0
1,1.613899,-0.252456,1.496094,1.496094,1.269531,-0.123132,1.178461,Gracy Singh,100_2.stl,44.0
2,1.617845,-0.254985,1.496094,1.496094,1.328125,-0.123911,1.126470,Gracy Singh,100_3.stl,44.0
3,1.597820,-0.254835,1.496094,1.496094,1.328125,-0.126173,1.126470,Gracy Singh,100_4.stl,44.0
4,1.592781,-0.253951,1.496094,1.496094,1.328125,-0.126333,1.126470,Gracy Singh,100_5.stl,44.0
...,...,...,...,...,...,...,...,...,...,...
144,1.525343,-0.226371,1.496094,1.496094,0.777344,-0.120163,1.924621,Divyanka,150_1.stl,61.0
145,1.562474,-0.230663,1.496094,1.496094,0.777344,-0.118102,1.924621,Divyanka,150_2.stl,61.0
146,1.547055,-0.228015,1.496094,1.496094,1.133256,-0.118496,1.320172,Divyanka,150_3.stl,61.0
147,1.550071,-0.228921,1.496094,1.496094,1.136719,-0.118620,1.316150,Divyanka,150_4.stl,61.0
